# Final Project TLF: Pipeline Klasifikasi & Clusterisasi Foto Sampah
## Arsitektur: MobileNetV2 (1280D) -> StandardScaler -> PCA -> FCM Manual (NumPy) -> KNN (+ mu)
**Dataset:** Kaggle "Garbage Images Dataset" (`zlatan599/garbage-dataset-classification`, Lisensi MIT)

Notebook ini dirancang untuk dijalankan di **Google Colab** atau **Jupyter Lab lokal** guna melatih model klasifikasi dan clusterisasi sampah terpadu:
- **Feature Extractor Utama:** MobileNetV2 pretrained ImageNet (output 1280 fitur float bersemantik tinggi), dengan fitur Handcrafted (HSV + GLCM ~26 dimensi) sebagai pembanding.
- **Anti Data Leakage:** Stratified train/test split (80:20) dilakukan **SEBELUM** fit StandardScaler, PCA, dan FCM.
- **Fuzzy C-Means (FCM) Manual:** Diimplementasikan dari dasar dengan NumPy murni. Sifat fuzzy menangani ambiguitas visual material sampah (misal: plastik transparan vs kaca, atau kardus kotor vs kertas).
- **Membership Data Baru/Test:** Menggunakan fungsi `fcm_predict_membership()` berdasarkan jarak ke pusat cluster hasil training tanpa fit ulang.
- **KNN Hybrid (+ mu):** Memadukan ruang representasi laten PCA dan derajat keanggotaan fuzzy $[\mathbf{X}_{PCA}, \boldsymbol{\mu} \times \text{mu\_weight}]$.
- **Dua Studi Komparasi:** (1) KNN Biasa vs KNN+mu FCM, dan (2) MobileNetV2 vs Handcrafted HSV+GLCM.
- **Ekspor Artefak:** Disimpan ke direktori `models/` (`scaler.pkl`, `pca.pkl`, `fcm_centers.npy`, `knn.pkl`, `meta.json`) untuk GUI Streamlit (`app.py`).


In [ ]:
# ==============================================================================
# 1. SETUP LINGKUNGAN & PATH (GOOGLE COLAB / LOKAL)
# ==============================================================================
import sys
import os

# Cek apakah dijalankan di lingkungan Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[*] Terdeteksi berjalan di Google Colab.")
    # Jika repository diletakkan di Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # %cd /content/drive/MyDrive/Garbage-Classification-FCM
    
    # Atau clone via git:
    # !git clone https://github.com/<username>/Garbage-Classification-FCM.git
    # %cd Garbage-Classification-FCM

    REPO_ROOT = os.path.abspath(".")
else:
    print("[*] Berjalan di lingkungan lokal / server.")
    REPO_ROOT = os.path.abspath("..")

# Tambahkan root proyek ke sys.path agar modul 'src' dapat diimport
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

print(f"[+] Root direktori proyek : {REPO_ROOT}")
print(f"[+] Status sys.path       : Modul 'src' siap diakses.")


In [ ]:
# ==============================================================================
# 2. PENGECEKAN VERSI DEPENDENSI (PINNING SCIKIT-LEARN & TENSORFLOW)
# ==============================================================================
import sklearn
import numpy as np
import joblib

try:
    import tensorflow as tf
    tf_version = tf.__version__
except ImportError:
    tf_version = "Belum terpasang (Jalankan: !pip install tensorflow>=2.15.0)"

EXPECTED_SKLEARN_VERSION = "1.4.2"

print("=" * 65)
print(f"Versi scikit-learn terpasang : {sklearn.__version__}")
print(f"Versi TensorFlow terpasang   : {tf_version}")
print(f"Versi NumPy terpasang        : {np.__version__}")
print(f"Versi Joblib terpasang       : {joblib.__version__}")
print("=" * 65)

if sklearn.__version__ != EXPECTED_SKLEARN_VERSION:
    print(f"[PERINGATAN] Versi scikit-learn ({sklearn.__version__}) tidak sama dengan {EXPECTED_SKLEARN_VERSION}.")
    print("Disarankan menjalankan perintah berikut di Colab:")
    print(f"!pip install scikit-learn=={EXPECTED_SKLEARN_VERSION}")
    print("Hal ini penting agar file .pkl yang disimpan dapat dimuat oleh aplikasi lokal Streamlit.")
else:
    print(f"[OK] Versi scikit-learn sesuai target ({EXPECTED_SKLEARN_VERSION}).")


## 3. Persiapan Dataset: Garbage Images Dataset
Dataset acuan proyek ini adalah **Garbage Images Dataset** di Kaggle:
- **Slug Dataset:** `zlatan599/garbage-dataset-classification` (Lisensi MIT)
- **Kategori / Kelas:** `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash` + `metadata.csv`

Jalankan sel di bawah ini untuk mengunduh dataset secara otomatis via Kaggle API di Colab atau gunakan skrip `data/download_dataset.py`.


In [ ]:
# ==============================================================================
# 3. PENGUNDUHAN DATASET VIA KAGGLE API DI GOOGLE COLAB
# ==============================================================================
# Jika belum mengonfigurasi kaggle.json di Colab, un-comment baris berikut:
# from google.colab import files
# files.upload() # Unggah file kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d zlatan599/garbage-dataset-classification -p data --unzip

DATA_DIR = os.path.join(REPO_ROOT, "data", "Garbage_Dataset_Classification", "images")
if not os.path.exists(DATA_DIR):
    alt_paths = [
        os.path.join(REPO_ROOT, "data", "Garbage Dataset Classification", "images"),
        os.path.join(REPO_ROOT, "data", "images"),
        os.path.join(REPO_ROOT, "data")
    ]
    for p in alt_paths:
        if os.path.exists(p) and any(os.path.isdir(os.path.join(p, d)) for d in os.listdir(p)):
            DATA_DIR = p
            break

print(f"Path dataset target: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    subdirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"[+] Dataset terdeteksi! Kategori subfolder: {subdirs}")
else:
    print("[INFO] Dataset belum tersedia di folder data/. Pipeline demonstrasi akan")
    print("       menggunakan sampel sintetis berbobot jika citra belum diunduh.")


## 4. Import Modul Inti dari `src/`
Sesuai aturan arsitektur sistem, seluruh logika ekstraksi fitur, FCM manual, dan pipeline training **wajib diimport dari modul `src/`**:
- `src.features`: `extract_features`, `get_mobilenet_model`, `extract_handcrafted_features`
- `src.fcm`: `FCM`, `fcm_predict_membership`
- `src.pipeline`: `fit_pipeline`, `transform`, `predict`, `save_models`, `load_models`, `compare_knn_variants`, `compare_feature_extractors`


In [ ]:
# ==============================================================================
# 4. IMPORT MODUL DARI SRC & LOAD CONFIG
# ==============================================================================
import yaml
from src.features import extract_features
from src.fcm import FCM, fcm_predict_membership
from src.pipeline import (
    fit_pipeline,
    transform,
    predict,
    save_models,
    load_models,
    compare_knn_variants,
    compare_feature_extractors
)

config_file = os.path.join(REPO_ROOT, "config.yaml")
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

print("[+] Konfigurasi default (config.yaml) berhasil dimuat:")
print(yaml.dump(config, default_flow_style=False))


---
## 5. CONTOH PERHITUNGAN MANUAL FUZZY C-MEANS (FCM) & PARTITION ENTROPY
### Simulasi Langkah demi Langkah: 5 Sampel x 2 Komponen PCA ($c=2, m=2.0$, 2 Iterasi Lengkap)

Salah satu syarat mutlak laporan dan tugas TLF adalah **setiap langkah algoritma harus dapat dibuktikan dan diverifikasi secara matematis**.
Berikut disajikan simulasi numerik manual FCM menggunakan:
- **Jumlah data ($N$):** 5 sampel (hasil proyeksi 2 komponen PCA $PC_1, PC_2$).
- **Jumlah cluster ($c$):** 2 cluster ($v_1, v_2$).
- **Fuzzifier ($m$):** 2.0.
- **Metrik Evaluasi Partisi:** Fuzzy Partition Coefficient (FPC) dan Partition Entropy (PE).

---

### Rumus Matematis:

1. **Pusat Cluster ($v_j$):**
$$v_j = \frac{\sum_{i=1}^N (u_{ij})^m x_i}{\sum_{i=1}^N (u_{ij})^m}, \quad j = 1, 2, \dots, c$$

2. **Jarak Euclidean ($d_{ij}$):**
$$d_{ij} = \|x_i - v_j\|_2 = \sqrt{\sum_{k=1}^p (x_{ik} - v_{jk})^2}$$

3. **Fungsi Objektif ($J$):**
$$J = \sum_{i=1}^N \sum_{j=1}^c (u_{ij})^m (d_{ij})^2$$

4. **Pembaruan Matriks Keanggotaan ($u_{ij}$):**
$$u_{ij} = \frac{1}{\sum_{k=1}^c \left(\frac{d_{ij}}{d_{ik}}\right)^{\frac{2}{m-1}}}$$
Untuk kasus khusus $c=2$ dan $m=2.0$:
$$u_{i1} = \frac{d_{i2}^2}{d_{i1}^2 + d_{i2}^2}, \quad u_{i2} = \frac{d_{i1}^2}{d_{i1}^2 + d_{i2}^2}$$

5. **Fuzzy Partition Coefficient (FPC):**
$$\text{FPC} = \frac{1}{N} \sum_{i=1}^N \sum_{j=1}^c (u_{ij})^2$$

6. **Partition Entropy (PE):**
$$\text{PE} = -\frac{1}{N} \sum_{i=1}^N \sum_{j=1}^c u_{ij} \ln(u_{ij}) \quad (\text{untuk } u_{ij} > 0)$$


In [ ]:
# ==============================================================================
# 5. PERHITUNGAN MANUAL FCM DENGAN FPC & PARTITION ENTROPY (PE)
# ==============================================================================
import numpy as np

# 5 Sampel data 2 Dimensi (PC1, PC2)
X_manual = np.array([
    [1.0, 1.5],  # Sampel 1 (Karakteristik Klaster A)
    [1.2, 1.8],  # Sampel 2
    [0.8, 1.2],  # Sampel 3
    [4.5, 4.0],  # Sampel 4 (Karakteristik Klaster B)
    [5.0, 4.5]   # Sampel 5
], dtype=float)

c = 2
m = 2.0
N = len(X_manual)

# Matriks Keanggotaan Awal U^(0) (setiap baris sum = 1.0)
U = np.array([
    [0.80, 0.20],
    [0.70, 0.30],
    [0.90, 0.10],
    [0.20, 0.80],
    [0.10, 0.90]
], dtype=float)

print("=" * 75)
print("       SIMULASI PERHITUNGAN MANUAL FUZZY C-MEANS (FCM) & ENTROPY")
print("=" * 75)
print(f"Jumlah sampel (N) : {N}")
print(f"Jumlah cluster (c): {c}")
print(f"Fuzzifier (m)     : {m}")

for iteration in range(1, 3):
    print(f"\n{'='*30} ITERASI {iteration} {'='*30}")
    
    # 1. Pembobot (u_ij)^m
    U_m = U ** m
    sum_U_m = np.sum(U_m, axis=0)
    
    # 2. Pusat Cluster v_j
    centers = np.dot(U_m.T, X_manual) / sum_U_m[:, np.newaxis]
    print(f"1. Pusat Cluster v (Iterasi {iteration}):")
    print(f"   Cluster 1 (v_1): PC1 = {centers[0, 0]:.4f}, PC2 = {centers[0, 1]:.4f}")
    print(f"   Cluster 2 (v_2): PC1 = {centers[1, 0]:.4f}, PC2 = {centers[1, 1]:.4f}")
    
    # 3. Matriks Jarak Euclidean d_ij
    D = np.zeros((N, c))
    for i in range(N):
        for j in range(c):
            D[i, j] = np.linalg.norm(X_manual[i] - centers[j])
            
    print(f"\n2. Matriks Jarak Euclidean D^(iter {iteration}) [||x_i - v_j||]:")
    print("   Sampel | Jarak ke v_1 (d_i1) | Jarak ke v_2 (d_i2)")
    for i in range(N):
        print(f"     x_{i+1}  |       {D[i, 0]:.4f}        |       {D[i, 1]:.4f}")
        
    # 4. Nilai Fungsi Objektif J
    J = np.sum(U_m * (D ** 2))
    print(f"\n3. Nilai Fungsi Objektif J^(iter {iteration}): {J:.6f}")
    
    # 5. Pembaruan Matriks Keanggotaan U
    U_new = np.zeros((N, c))
    p = 2.0 / (m - 1.0)
    for i in range(N):
        for j in range(c):
            denom = sum((D[i, j] / D[i, k]) ** p for k in range(c))
            U_new[i, j] = 1.0 / denom
            
    # 6. Evaluasi FPC dan Partition Entropy (PE)
    fpc = np.sum(U_new ** 2) / N
    # PE = - (1/N) * sum(u_ij * ln(u_ij))
    eps = 1e-12
    pe = -np.sum(U_new * np.log(np.maximum(U_new, eps))) / N
    
    print(f"\n4. Matriks Keanggotaan Diperbarui U^(iter {iteration}):")
    print("   Sampel | Cluster 1 (u_i1) | Cluster 2 (u_i2) | Partisi Crisp (Cluster)")
    for i in range(N):
        cluster_assignment = np.argmax(U_new[i]) + 1
        print(f"     x_{i+1}  |      {U_new[i, 0]:.4f}      |      {U_new[i, 1]:.4f}      | Cluster {cluster_assignment}")
        
    print(f"\n5. Metrik Evaluasi Partisi (Iterasi {iteration}):")
    print(f"   - Fuzzy Partition Coefficient (FPC) : {fpc:.6f}  (Mendekati 1.0 = partisi semakin tegas)")
    print(f"   - Partition Entropy (PE)            : {pe:.6f}  (Mendekati 0.0 = ketidakpastian semakin rendah)")
    
    U = U_new

print("\n" + "=" * 75)
print("[VERIFIKASI NUMERIK SELESAI] Konvergensi dan penurunan entropi terbukti secara matematis.")
print("=" * 75)


---
## 6. Ekstraksi Fitur Citra
Fitur diekstraksi dari citra sampel subfolder:
- **MobileNetV2 (1280 Dimensi):** Fitur utama berbasis deep representation ImageNet.
- **Handcrafted (26 Dimensi):** HSV histogram (16) + statistik warna (6) + GLCM (4) sebagai baseline pembanding.


In [ ]:
# ==============================================================================
# 6. PEMUATAN CITRA & EKSTRAKSI FITUR
# ==============================================================================
import glob
from PIL import Image

def load_dataset_features(data_dir, extractor_type="mobilenetv2", samples_per_class=60):
    X_list = []
    y_list = []
    class_names = []
    
    if os.path.exists(data_dir):
        subdirs = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        if len(subdirs) >= 6:
            class_names = subdirs
            print(f"[*] Mengekstrak fitur '{extractor_type}' dari {len(subdirs)} subfolder kelas...")
            for cls in subdirs:
                cls_folder = os.path.join(data_dir, cls)
                img_paths = glob.glob(os.path.join(cls_folder, "*.jpg")) +                             glob.glob(os.path.join(cls_folder, "*.jpeg")) +                             glob.glob(os.path.join(cls_folder, "*.png"))
                selected = img_paths[:samples_per_class]
                print(f"    - Kelas '{cls}': {len(selected)} citra diproses.")
                for p in selected:
                    feats = extract_features(p, feature_extractor=extractor_type, config=config)
                    X_list.append(feats)
                    y_list.append(cls)
            return np.array(X_list), np.array(y_list), class_names

    # Fallback sintetis jika dataset belum diunduh (menjamin notebook selalu dapat dieksekusi)
    print(f"[INFO] Citra mentah belum tersedia di {data_dir}.")
    print(f"       Membuat data sintetis {extractor_type} untuk validasi end-to-end...")
    class_names = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
    n_dim = 1280 if extractor_type == "mobilenetv2" else 26
    np.random.seed(42)
    for idx, cls in enumerate(class_names):
        cluster_mean = np.random.uniform(0.1, 0.9, size=n_dim) + (idx * 0.25)
        feats = np.random.normal(loc=cluster_mean, scale=0.08, size=(samples_per_class, n_dim))
        for f in feats:
            X_list.append(f)
            y_list.append(cls)
            
    return np.array(X_list), np.array(y_list), class_names

X_raw, y_raw, class_names = load_dataset_features(DATA_DIR, extractor_type="mobilenetv2", samples_per_class=50)
print(f"\n[+] Dimensi Fitur X : {X_raw.shape}")
print(f"[+] Total Label y   : {y_raw.shape}")
print(f"[+] Daftar Kelas    : {class_names}")


---
## 7. Pelatihan Pipeline Terpadu (Anti Data Leakage)
Langkah pelatihan:
1. **Stratified Split (80% Train, 20% Test)** dilakukan pertama kali.
2. `StandardScaler` di-fit **hanya** pada `X_train`.
3. `PCA` di-fit **hanya** pada `X_train_scaled` (ambang variansi 90%, max 50 komponen).
4. `FCM` manual di-fit **hanya** pada `X_train_pca`.
5. Derajat keanggotaan data uji $\mu_{test}$ dihitung dengan `fcm_predict_membership()`.
6. `KNN` dilatih pada input gabungan $[\mathbf{X}_{PCA}, \boldsymbol{\mu} \times \text{mu\_weight}]$.
7. Evaluasi model pada test set terpisah.


In [ ]:
# ==============================================================================
# 7. TRAINING PIPELINE TERPADU DENGAN FIT_PIPELINE()
# ==============================================================================
print("[*] Melatih pipeline melalui fit_pipeline()...")
results = fit_pipeline(
    X_raw,
    y_true=y_raw,
    config=config,
    test_size=0.2,
    random_state=42
)

scaler = results["scaler"]
pca = results["pca"]
fcm_model = results["fcm"]
knn = results["knn"]
X_pca_train = results["X_pca_train"]
X_pca_test = results["X_pca_test"]
metrics = results.get("metrics", {})

print("\n[+] Pipeline Berhasil Dilatih!")
print(f"    - Komponen PCA terpilih : {pca.n_components_}")
print(f"    - Explained Variance    : {np.sum(pca.explained_variance_ratio_):.4f}")
print(f"    - FPC Score             : {results.get('fpc', 'N/A'):.4f}")
print(f"    - Partition Entropy(PE) : {results.get('pe', 'N/A'):.4f}")
print(f"    - Silhouette Score      : {results.get('silhouette', 'N/A'):.4f}")
if "accuracy" in metrics:
    print(f"    - Test Accuracy         : {metrics['accuracy']:.4f}")
    print(f"    - Test F1-Score (macro) : {metrics['f1_macro']:.4f}")


---
## 8. Evaluasi & Visualisasi Hasil
Visualisasi:
1. Kurva konvergensi fungsi objektif $J$ FCM.
2. Scatter plot sebaran cluster pada 2 Komponen Utama PCA ($PC_1$ vs $PC_2$) beserta posisi pusat cluster FCM.
3. Confusion matrix evaluasi KNN terhadap label kelas aktual.


In [ ]:
# ==============================================================================
# 8. VISUALISASI HASIL PELATIHAN & EVALUASI
# ==============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Konvergensi Fungsi Objektif J FCM
axes[0].plot(range(1, len(fcm_model.objective_history_) + 1), fcm_model.objective_history_, marker='o', color='crimson')
axes[0].set_title("Konvergensi Fungsi Objektif J FCM", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Iterasi ke-")
axes[0].set_ylabel("Nilai Fungsi Objektif J")
axes[0].grid(True, linestyle="--", alpha=0.6)

# Plot 2: Scatter Plot PC1 vs PC2
scatter = axes[1].scatter(X_pca_train[:, 0], X_pca_train[:, 1], c=results["cluster_labels_train"], cmap='viridis', alpha=0.6, edgecolors='k')
centers_pca = fcm_model.centers_
axes[1].scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Pusat FCM')
axes[1].set_title("Proyeksi Data Training pada PC1 vs PC2", fontsize=12, fontweight='bold')
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

# Tampilkan Confusion Matrix jika tersedia
if "confusion_matrix" in metrics and "y_test" in results:
    cm = metrics["confusion_matrix"]
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=results.get("classes", class_names))
    fig, ax = plt.subplots(figsize=(7, 6))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title("Confusion Matrix pada Data Uji (Test Set)", fontweight="bold")
    plt.show()


---
## 9. Studi Komparasi Terpadu (Dua Perbandingan)

Sesuai spesifikasi proyek, aplikasi menguji dua perbandingan penting:
1. **Perbandingan Model:** KNN Biasa (hanya fitur PCA) vs KNN + mu FCM (fitur PCA + derajat keanggotaan fuzzy $\mu$).
2. **Perbandingan Ekstraktor Fitur:** MobileNetV2 (1280 dimensi) vs Handcrafted (HSV + GLCM 26 dimensi).


In [ ]:
# ==============================================================================
# 9. STUDI KOMPARASI (KNN BIASA VS KNN+MU, & MOBILENETV2 VS HANDCRAFTED)
# ==============================================================================
print("=" * 70)
print("  STUDI KOMPARASI 1: KNN BIASA vs KNN + MU FCM")
print("=" * 70)
df_comp_knn = compare_knn_variants(
    X_train_pca=X_pca_train,
    u_train=results["u_train"],
    y_train=results["y_train"],
    X_test_pca=X_pca_test,
    u_test=results["u_test"],
    y_test=results["y_test"],
    mu_weight=config.get("knn", {}).get("mu_weight", 1.0),
    k=config.get("knn", {}).get("n_neighbors", 5)
)
print(df_comp_knn.to_string(index=False))

print("\n" + "=" * 70)
print("  STUDI KOMPARASI 2: MOBILENETV2 vs HANDCRAFTED (HSV + GLCM)")
print("=" * 70)
df_comp_ext = compare_feature_extractors(
    data_dir=DATA_DIR,
    config=config,
    samples_per_class=30
)
print(df_comp_ext.to_string(index=False))


---
## 10. Penyimpanan Artefak Model ke Direktori `models/`
Menyimpan 5 artefak:
1. `scaler.pkl`: StandardScaler
2. `pca.pkl`: Model reduksi dimensi PCA
3. `fcm_centers.npy`: Koordinat pusat cluster FCM di ruang PCA
4. `knn.pkl`: Model klasifikasi KNN terlatih pada $[\text{PCA}, \mu \times \text{mu\_weight}]$
5. `meta.json`: Metadata konfigurasi, nama kelas dinamis, versi library, dan bobot $\mu$.

File-file ini akan digunakan oleh antarmuka Streamlit (`app.py`) untuk inferensi langsung tanpa perlu training ulang.


In [ ]:
# ==============================================================================
# 10. SIMPAN ARTEFAK & VERIFIKASI PEMUATAN KEMBALI
# ==============================================================================
MODELS_DIR = os.path.join(REPO_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Simpan artefak resmi
save_models(
    models_dir=MODELS_DIR,
    scaler=scaler,
    pca=pca,
    fcm_centers=fcm_model.centers_,
    knn=knn,
    config=config,
    class_names=results.get("classes", class_names),
    feature_extractor=config.get("features", {}).get("feature_extractor", "mobilenetv2"),
    mu_weight=config.get("knn", {}).get("mu_weight", 1.0)
)

print(f"[+] Artefak berhasil disimpan di: {MODELS_DIR}")
print(f"[+] Daftar file artefak: {os.listdir(MODELS_DIR)}")

# Verifikasi pemuatan kembali
print("\n[*] Memverifikasi pemanggilan load_models()...")
loaded = load_models(models_dir=MODELS_DIR)
if loaded is not None:
    print("[SUCCESS] Seluruh 5 artefak model berhasil dimuat kembali secara valid!")
    print(f"          - Scaler     : {type(loaded['scaler']).__name__}")
    print(f"          - PCA        : {type(loaded['pca']).__name__} (komponen: {loaded['pca'].n_components_})")
    print(f"          - FCM Centers: shape {loaded['fcm_centers'].shape}")
    print(f"          - KNN        : {type(loaded['knn']).__name__}")
    print(f"          - Metadata   : {loaded.get('meta', {})}")
else:
    print("[ERROR] Gagal memuat artefak model.")
